# Traces and Evaluation — EcoTravel Agent

This notebook generates 5 evaluated interaction traces, including an LLM comparison trace.
All traces are sent to LangSmith project `eco-travel-agent`.

## Evaluation Approach
- **Traces 1–4**: Standard interactions with `claude-sonnet-4-6`
- **Trace 5**: Side-by-side comparison of `claude-sonnet-4-6` vs `claude-haiku-4-5-20251001` on the same query
- **Evaluation method**: LLM-as-judge (Claude Haiku scores each response on 5 dimensions)
- **ROI analysis**: Cost vs quality comparison to recommend the production model

In [ ]:
import os
import sys
import json
import anthropic
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / ".env")

from src.agent import EcoTravelAgent
from src.models import UserPreferences
from src.tracing import traced_chat

BASE_PREFS = UserPreferences(
    budget_per_night=200.0,
    weather_preference="warm",
    max_drive_miles=25,
    crowd_tolerance="low",
)

print("Setup complete. Base preferences:")
print(f"  Budget: ${BASE_PREFS.budget_per_night:.0f}/night, Crowd tolerance: {BASE_PREFS.crowd_tolerance}")

## Trace 1: Hotel Search with Preference Filtering

In [ ]:
agent1 = EcoTravelAgent(model="claude-sonnet-4-6")
agent1.memory.set_preferences(BASE_PREFS)

trace1 = traced_chat(
    agent1,
    "Find hotels in Savannah, GA from July 10-14 within 25 miles",
    run_name="trace-1-search"
)
print(trace1)

## Trace 2: Hotel Details + Review Summary

In [ ]:
trace2 = traced_chat(
    agent1,
    "Tell me more about the top hotel result and its guest reviews",
    run_name="trace-2-details"
)
print(trace2)

## Trace 3: Nearby Low-Crowd Destinations

In [ ]:
trace3 = traced_chat(
    agent1,
    "Find nearby quieter towns I could explore instead of Savannah",
    run_name="trace-3-nearby"
)
print(trace3)

## Trace 4: Low-Crowd Itinerary

In [ ]:
trace4 = traced_chat(
    agent1,
    "Build me a low-crowd itinerary for my Savannah stay",
    run_name="trace-4-itinerary"
)
print(trace4)

## Trace 5: LLM Comparison — Sonnet vs Haiku

The same query is sent to both models. The LLM-as-judge evaluates both responses.

In [ ]:
COMPARISON_QUERY = (
    "Find low-crowd hotels in Sedona, AZ from August 5-8 within 25 miles. "
    "Explain the crowd factors for the top result."
)

# Sonnet
agent_sonnet = EcoTravelAgent(model="claude-sonnet-4-6")
agent_sonnet.memory.set_preferences(BASE_PREFS)
trace5_sonnet = traced_chat(agent_sonnet, COMPARISON_QUERY, run_name="trace-5-sonnet")

# Haiku
agent_haiku = EcoTravelAgent(model="claude-haiku-4-5-20251001")
agent_haiku.memory.set_preferences(BASE_PREFS)
trace5_haiku = traced_chat(agent_haiku, COMPARISON_QUERY, run_name="trace-5-haiku")

print("=== SONNET RESPONSE ===")
print(trace5_sonnet)
print("\n=== HAIKU RESPONSE ===")
print(trace5_haiku)

## LLM-as-Judge Evaluation

Claude Haiku scores each response on 5 dimensions (1–5 scale):
- **relevance**: Does it address the travel query?
- **crowd_focus**: Does it emphasize low-crowd recommendations?
- **detail_quality**: Are the details specific and actionable?
- **tone**: Is it helpful and professional?
- **overall**: Overall response quality

In [ ]:
judge_client = anthropic.Anthropic()

def llm_judge(query: str, response: str, model_name: str) -> dict:
    prompt = f"""You are evaluating a travel agent AI response. Score each dimension 1-5.

Query: {query}
Model: {model_name}
Response: {response}

Scoring criteria:
- relevance (1-5): Does the response directly address the travel query?
- crowd_focus (1-5): Does it emphasize low-crowd recommendations as instructed?
- detail_quality (1-5): Are the details specific and actionable?
- tone (1-5): Is it helpful, professional, and concise?
- overall (1-5): Overall quality

Return ONLY valid JSON with no extra text:
{{"relevance": N, "crowd_focus": N, "detail_quality": N, "tone": N, "overall": N, "notes": "brief comment"}}"""

    result = judge_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    return json.loads(result.content[0].text)

scores_sonnet = llm_judge(COMPARISON_QUERY, trace5_sonnet, "claude-sonnet-4-6")
scores_haiku = llm_judge(COMPARISON_QUERY, trace5_haiku, "claude-haiku-4-5-20251001")

comparison_df = pd.DataFrame([
    {"Model": "claude-sonnet-4-6", **scores_sonnet},
    {"Model": "claude-haiku-4-5-20251001", **scores_haiku},
]).set_index("Model")

display(comparison_df)

## ROI Analysis — Model Cost vs Quality

In [ ]:
# Pricing (USD per million tokens) — verify at anthropic.com/pricing
PRICING = {
    "claude-sonnet-4-6":         {"input": 3.00,  "output": 15.00},
    "claude-haiku-4-5-20251001": {"input": 0.80,  "output":  4.00},
}

# Usage estimate: 500 sessions/day, ~2,500 input + 600 output tokens per session
DAILY_SESSIONS    = 500
AVG_INPUT_TOKENS  = 2500
AVG_OUTPUT_TOKENS = 600
DAYS_PER_MONTH    = 30

def monthly_cost(model_id: str) -> float:
    p = PRICING[model_id]
    per_session = (
        (AVG_INPUT_TOKENS  / 1_000_000 * p["input"]) +
        (AVG_OUTPUT_TOKENS / 1_000_000 * p["output"])
    )
    return per_session * DAILY_SESSIONS * DAYS_PER_MONTH

cost_sonnet = monthly_cost("claude-sonnet-4-6")
cost_haiku  = monthly_cost("claude-haiku-4-5-20251001")
cost_ratio  = cost_sonnet / cost_haiku

sonnet_overall = scores_sonnet["overall"]
haiku_overall  = scores_haiku["overall"]
quality_lift   = (sonnet_overall - haiku_overall) / max(haiku_overall, 1)

print(f"Monthly cost — Sonnet:  ${cost_sonnet:,.2f}")
print(f"Monthly cost — Haiku:   ${cost_haiku:,.2f}")
print(f"Cost multiplier (Sonnet vs Haiku): {cost_ratio:.1f}x")
print(f"Quality lift (Sonnet over Haiku):  {quality_lift:.1%}")
print()

if quality_lift / max(cost_ratio - 1, 0.01) > 0.15:
    recommendation = "claude-sonnet-4-6"
    rationale = (
        "The quality improvement justifies the cost premium "
        "for a user-facing travel recommendation product."
    )
else:
    recommendation = "claude-haiku-4-5-20251001"
    rationale = (
        "The cost savings outweigh the marginal quality difference at this volume. "
        "Use Haiku in production, Sonnet for complex itinerary generation."
    )

print(f"Recommended production model: {recommendation}")
print(f"Rationale: {rationale}")

roi_df = pd.DataFrame([
    {"Model": "claude-sonnet-4-6", "Monthly Cost ($)": round(cost_sonnet, 2),
     "Overall Score": sonnet_overall, "Notes": scores_sonnet.get("notes", "")},
    {"Model": "claude-haiku-4-5-20251001", "Monthly Cost ($)": round(cost_haiku, 2),
     "Overall Score": haiku_overall, "Notes": scores_haiku.get("notes", "")},
]).set_index("Model")
display(roi_df)

## Performance Commentary

*Fill in after running all cells above with your actual observations.*

### Overall Agent Performance

[Describe how the agent performed across traces 1–4. Did it correctly surface crowd scores?
Were preference filters applied? Was the output format clear and actionable?]

### What the Evaluation Showed

[Describe patterns from LLM judge scores. Were crowd_focus and detail_quality strong or weak?
Any surprising results from the scoring?]

### Sonnet vs Haiku Comparison

[Describe specific differences observed between the two model responses.
Reference the ROI analysis to give a final production model recommendation.]

In [ ]:
print("=== Evaluation Summary ===")
print(f"Traces generated: 5 (4 sonnet standard + 1 model comparison)")
print(f"Evaluation method: LLM-as-judge (claude-haiku-4-5-20251001)")
print(f"LangSmith project: {os.environ.get('LANGCHAIN_PROJECT', 'eco-travel-agent')}")
print(f"\nTrace names:")
for name in ["trace-1-search", "trace-2-details", "trace-3-nearby",
             "trace-4-itinerary", "trace-5-sonnet", "trace-5-haiku"]:
    print(f"  - {name}")
print(f"\nView traces at: https://smith.langchain.com")